In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [8]:
def generate_fbm(n,H,T=0.1):
    t=np.linspace(0,T,n)
    sigma=np.zeros((n,n))

    for i in range(n):
        for j in range(n):
            if i==0 and j==0:
                sigma[i,j]=0.0
            else:
                sigma[i,j]=0.5*((t[i]**(2*H))+(t[j]**(2*H))-np.abs((t[i]-t[j])**(2*H)))
                        
    sigma+=np.eye(n)*1e-9
    L=np.linalg.cholesky(sigma)
    Z=np.random.normal(0,1,n)

    W=L@Z
    return t,W


In [11]:
np.random.seed(42)
H=0.1
t,W_h=generate_fbm(500,H)


/var/folders/gj/hjmxx9yn1q3d9y386xr_f6bw0000gn/T/ipykernel_17879/2529873187.py:10: RuntimeWarning: invalid value encountered in scalar power
  sigma[i,j]=0.5*((t[i]**(2*H))+(t[j]**(2*H))-np.abs((t[i]-t[j])**(2*H)))


In [12]:
eta=0.15
volatility=0.2*np.exp(eta*W_h-0.5*(eta**2)*(t**(2*H)))
log_volatility=np.log(volatility)

In [20]:
deltas=np.arange(1,50,2)

moments=[]
for d in deltas:
    diff=log_volatility[d:]-log_volatility[:-d]
    m2=np.mean(abs(diff)**3)
    moments.append(m2)


In [21]:
log_deltas=np.log(deltas)
log_moments=np.log(moments)


In [22]:
X = sm.add_constant(log_deltas)
model = sm.OLS(log_moments, X).fit()
slope = model.params[1]
H_estimated = slope / 3

print(f"True H: {H}")
print(f"Estimated H from regression: {H_estimated:.4f}")

True H: 0.1
Estimated H from regression: 0.0819
